# rescore8b_run — 8B chấm lại DANH SÁCH RÚT GỌN, đo TỈ LỆ PHÁ trên trọn dev300

**Vì sao:** `hard15` cho `Qwen3-Reranker-8B` = **10/15**, **trùng khít tập 10 câu của
Gemini**, và **bao trùm hẳn** `ce_ours` (6/15) — thêm 4 câu, không mất câu nào. Tín hiệu
mạnh nhất tìm được từ đầu giải. **Nhưng hard15 chỉ đo phần ĂN THÊM.** Gemini cũng 10/15
mà đem thay toàn bộ thì **lỗ 2 câu** (19/08). Chưa có tỉ lệ phá thì con số 10/15 vô dụng.

**Vì sao RÚT GỌN chứ không chấm cả rổ:** 8B chạy **4,69 giây/cặp** (đo thật, 58,6 phút
cho 750 cặp — chậm hơn AITeamVN **37 lần**). Chấm trọn rổ 50 trên dev300 = 15.000 cặp =
**19,5h**, vượt trần 12h/lượt. Nhưng vai trò định dùng nó vốn **đã là** chấm lại danh sách
ngắn ("3 mình + 2 model ngoài", 19/08), nên đo trên top-10 **không phải cắt xén — đó
chính là cấu hình sẽ chạy thật.**

| việc | cặp | giờ |
|---|---|---|
| dev300 top-10 ← **lượt này** | 3.000 | **3,9h** |
| đề thi top-8 (nếu thắng) | 8.000 | 10,4h — vừa dưới trần |
| ~~dev300 cả rổ 50~~ | 15.000 | ~~19,5h~~ vượt trần |

### ⚠️ TRẦN CỦA CẢ HƯỚNG NÀY LÀ **+2,17 ĐIỂM** — đo trên CPU 23/08, đọc trước khi đặt ngưỡng

Nếu 8B chọn **hoàn hảo** 5 văn bản trong top-10 thì dev300 đạt **0.9567**, tức hơn mốc
0.9350 đúng **+2,17**. Không ai vượt được con số đó. Trần theo cỡ danh sách:

| TOPN | trần oracle | dư địa | cặp | giờ 8B | |
|---|---|---|---|---|---|
| **10** | 0.9567 | **+2,17** | 3.000 | **3,9h** | ← chọn: dư địa/giờ tốt nhất |
| 15 | 0.9567 | +2,17 | 4.500 | 5,9h | y hệt trần mà đắt hơn 50% — bỏ |
| 20 | 0.9600 | +2,50 | 6.000 | 7,8h | thêm +0,33 trần, giá gấp đôi |
| 30 | 0.9717 | +3,67 | 9.000 | 11,7h | sát trần 12h/lượt, quá rủi ro |
| 50 | 0.9817 | +4,67 | 15.000 | 19,5h | vượt trần |

**Hệ quả phải nuốt:** kể cả hoàn hảo, chấm lại danh sách ngắn chỉ đóng được **hơn nửa**
khoảng cách 3,9 điểm tới đỉnh bảng. Đây không phải viên đạn bạc.

### Ngưỡng đặt TRƯỚC — **đã hiệu chỉnh theo trần, không dùng luật +2,0 thẳng**

Luật "≥+2,0 mới mang lên đề thi" được đặt cho hiệu ứng có trần mở. Ở đây trần chỉ +2,17,
nên đòi +2,0 là đòi 92% oracle — bất khả. Ngưỡng mới, đặt **trước** khi chạy, dựa trên
trần đã tính **trước** khi chạy:

| ra | quyết định |
|---|---|
| **Δ ≥ +1,50 VÀ ròng ≥ +4 câu** | ~70% oracle → **đổi CƠ CHẾ thật** → chạy đề thi top-8 |
| +0,70 … +1,50 | **vùng mù** → KHÔNG mang lên đề thi |
| ≤ +0,70 | đóng vai trò này |
| **tỉ lệ phá > 3%** | **ĐÓNG bất kể điểm ròng** — đúng chế độ hỏng đã giết Gemini (4,3%) |

Vì sao dám hạ xuống +1,50: +1,50 = 4,5 câu = **3 lần** sàn nhiễu ±1,5 câu của dev300. Và
đây là **đổi cơ chế**, không phải vặn tham số — hai lần đổi cơ chế trước (deepchunk, rổ
fusion) đều **vượt** dự báo trên public (134%, 192%), trong khi vặn heuristic thì hụt (31%).

**Bước 3 dò tốc độ trước** trên 100 cặp: `bf16` (mốc) vs `fp16` vs batch lớn hơn. T4 không
có bf16 phần cứng nên torch chạy đường giả lập — nghi đây là chỗ mất phần lớn thời gian.
Mỗi cấu hình vẫn qua `sanity check` của `load_qwen_reranker`, nên fp16 mà tràn số thì bị
chặn ngay chứ không âm thầm ra điểm rác (đúng bẫy đã dính với Qwen3-0.6B).

**Upload thêm ĐÚNG MỘT file:** `Ketqua_E/scores_dev300_fusion_M20_K20.json` (1,2MB) —
lượt này cần bảng xếp hạng hiện tại để biết top-10 là những văn bản nào. Còn lại
(`deep_chunk.py`, `rerank*.py`, `fusion_rrf_top50_dev_1000.json`, `dev_300_locked.json`,
`selected-contexts`) đã có sẵn.

> 🔴 Bật GPU. Bước 1 có assert chặn.


In [ ]:
!pip install -q -U "transformers>=4.51" bitsandbytes accelerate sentence-transformers

In [ ]:
# ===== Bước 1: dựng danh sách rút gọn top-10 theo bảng xếp hạng HIỆN TẠI =====
import os, sys, json, time, hashlib, gc
import torch
assert torch.cuda.is_available(), "KHÔNG CÓ GPU. Settings -> Accelerator -> GPU rồi chạy lại."
print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL      = "Qwen/Qwen3-Reranker-8B"
TOPN       = 10                     # số văn bản/câu đưa cho 8B chấm lại
HEAD, EXC  = 260, 900               # y hệt hard15 -> so sánh được
MOC        = 0.9350                 # max n=1 trên rổ fusion
GIO_TRAN   = 9.0                    # ước quá số này thì DỪNG, đừng đốt lượt

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

b = open(f"{INPUT_DIR}/rerank_qwen.py", "rb").read()
print(f"rerank_qwen.py {len(b)} bytes {hashlib.sha256(b).hexdigest()[:12]}")
import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800

SRC = f"{INPUT_DIR}/scores_dev300_fusion_M20_K20.json"
assert os.path.isfile(SRC), f"CHƯA UPLOAD {SRC} — lượt này cần bảng xếp hạng hiện tại"
S    = json.load(open(SRC, encoding="utf-8"))
dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
cand = json.load(open(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000.json", encoding="utf-8"))
qids = [q for q in dev if q in S]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qids}
ORDER= {q: [str(c["doc_id"]) for c in
            sorted(cand[q], key=lambda c: -float(c["rrf_score"]))] for q in qids}

# Dựng lại mốc 0.9350 NGAY TẠI ĐÂY. Lệch là sai chỗ nạp -> dừng, đừng chấm gì cả.
OURS = {q: DC.rank_by(S[q], "max") for q in qids}
rec  = lambda pred: sum(len(GOLD[q] & set(pred[q])) / len(GOLD[q]) for q in qids) / len(qids)
base = {n: rec({q: blend_bm25_first(OURS[q], ORDER[q], k=5, n_bm25=n) for q in qids})
        for n in (0, 1, 2, 3)}
print("\ndựng lại mốc:", {n: round(v, 4) for n, v in base.items()},
      "   CLAUDE.md ghi 0.9283 / 0.9350 / 0.9283 / 0.9233")
assert abs(base[1] - MOC) < 1e-3, f"KHÔNG dựng lại được mốc {MOC} -> sai chỗ nạp, DỪNG"

t0 = time.time()
short, pairs, index = {}, [], []
for i, q in enumerate(qids, 1):
    qs = dev[q]["question"]
    short[q] = OURS[q][:TOPN]
    for d in short[q]:
        e = DC.pick_chunks(qs, CTX_DIR, d, k=1)
        txt = DC.read_passage(CTX_DIR, d)[:HEAD].strip() + "\n\n" + (e[0] if e else "")[:EXC]
        pairs.append([qs, txt]); index.append((q, d))
    if i % 100 == 0: print(f"  băm {i}/{len(qids)} | {(time.time()-t0)/60:.1f} phút", flush=True)

print(f"\n{len(qids)} câu × top-{TOPN} = {len(pairs):,} cặp")
ORACLE = rec({q: sorted(short[q], key=lambda d: d not in GOLD[q])[:5] for q in qids})
print(f"gold nằm trong top-{TOPN} ở {sum(1 for q in qids if GOLD[q] & set(short[q]))}/{len(qids)} câu")
print(f"TRẦN oracle (8B chọn hoàn hảo 5 trong {TOPN}) = {ORACLE:.4f}"
      f"  ->  dư địa TỐI ĐA {(ORACLE-MOC)*100:+.2f} điểm. Không ai vượt được.")
assert abs(ORACLE - 0.9567) < 1e-3, "trần lệch so với bản tính CPU 23/08 -> soi lại chỗ nạp"

In [ ]:
# ===== Bước 2: dò tốc độ trên 100 cặp — bf16 (mốc) vs fp16 vs batch lớn =====
import rerank_qwen as RQ
PROBE = pairs[:100]
CFGS = [("bf16 bs4  (mốc lượt trước)", dict(dtype=torch.bfloat16, batch_size=4)),
        ("bf16 bs16",                  dict(dtype=torch.bfloat16, batch_size=16)),
        ("fp16 bs16",                  dict(dtype=torch.float16,  batch_size=16))]

speed = {}
for name, kw in CFGS:
    try:
        m = RQ.load_qwen_reranker(MODEL, device="cuda", load_4bit=True, max_length=1536, **kw)
        t = time.time(); m.predict(PROBE); r = (time.time() - t) / len(PROBE)
        speed[name] = (r, kw)
        print(f"  >>> {name:28s} {r:.2f} s/cặp · dev300 top-{TOPN} ước {r*len(pairs)/3600:.1f}h\n")
    except Exception as e:
        print(f"  >>> {name:28s} HỎNG — {type(e).__name__}: {str(e)[:200]}\n")
    finally:
        globals().pop("m", None); gc.collect(); torch.cuda.empty_cache()

assert speed, "không cấu hình nào chạy được"
best, (rate, KW) = min(speed.items(), key=lambda x: x[1][0])
est = rate * len(pairs) / 3600
print(f"CHỌN: {best} · {rate:.2f} s/cặp · ước {est:.1f}h cho {len(pairs):,} cặp")
assert est < GIO_TRAN, (f"ước {est:.1f}h > trần {GIO_TRAN}h. Hạ TOPN xuống "
                        f"{int(GIO_TRAN*3600/rate/len(qids))} rồi chạy lại. DỪNG.")

In [ ]:
# ===== Bước 3: chấm 3.000 cặp bằng cấu hình nhanh nhất =====
m = RQ.load_qwen_reranker(MODEL, device="cuda", load_4bit=True, max_length=1536, **KW)
t0 = time.time()
sc = m.predict(pairs)
print(f"xong {(time.time()-t0)/60:.1f} phút · nhịp thật {(time.time()-t0)/len(pairs):.2f} s/cặp")

R8 = {}
for (q, d), v in zip(index, sc):
    R8.setdefault(q, {})[d] = float(v)
p = f"{OUT}/scores_dev300_8b_top{TOPN}.json"
json.dump(R8, open(p, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p} — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN (quy tắc 2)")
globals().pop("m", None); gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ===== Bước 4: quét cách ghép trên CPU, đọc theo ngưỡng đã đặt trước =====
R8ORD = {q: sorted(R8[q], key=lambda d: -R8[q][d]) for q in qids}

def ghep(q, k):
    """k suất đầu của MÌNH, phần còn lại theo 8B. k=5 là bài cũ, k=0 là thay hẳn."""
    out = list(dict.fromkeys(OURS[q][:k]))
    for d in R8ORD[q]:
        if len(out) >= 5: break
        if d not in out: out.append(d)
    return out[:5]

print(f"{'cách chốt':26s}" + "".join(f"   n={n}  " for n in (0, 1, 2, 3)))
tab = {}
for k in range(6):
    lab = "thay hẳn bằng 8B" if k == 0 else ("bài mình (mốc)" if k == 5 else f"{k} mình + {5-k} của 8B")
    tab[k] = {n: rec({q: blend_bm25_first(ghep(q, k), ORDER[q], k=5, n_bm25=n) for q in qids})
              for n in (0, 1, 2, 3)}
    print(f"{lab:26s}" + "".join(f"  {tab[k][n]:.4f}" for n in (0, 1, 2, 3)))

bk, bn = max(((k, n) for k in tab for n in tab[k]), key=lambda x: tab[x[0]][x[1]])
got = tab[bk][bn]
# đối chiếu từng câu với bài chốt -> cứu/phá, số câu chứ không phải điểm
cur = {q: blend_bm25_first(OURS[q], ORDER[q], k=5, n_bm25=1) for q in qids}
new = {q: blend_bm25_first(ghep(q, bk), ORDER[q], k=5, n_bm25=bn) for q in qids}
cuu = [q for q in qids if (GOLD[q] & set(new[q])) and not (GOLD[q] & set(cur[q]))]
pha = [q for q in qids if (GOLD[q] & set(cur[q])) and not (GOLD[q] & set(new[q]))]

print("\n" + "=" * 64)
print(f"tốt nhất: k={bk} · n={bn} · {got:.4f}   (mốc {MOC})   Δ {(got-MOC)*100:+.2f} điểm")
print(f"cứu {len(cuu)} câu · phá {len(pha)} câu · ròng {len(cuu)-len(pha):+d} câu")
print(f"  phá: {pha[:12]}")
print(f"TỈ LỆ PHÁ = {len(pha)}/{sum(1 for q in qids if GOLD[q] & set(cur[q]))} câu đang đúng"
      f" = {100*len(pha)/max(1,sum(1 for q in qids if GOLD[q] & set(cur[q]))):.1f}%"
      f"   (Gemini 19/08: 4,3% -> lỗ ròng)")
print("=" * 64)
d, net = (got - MOC) * 100, len(cuu) - len(pha)
pct = 100 * (got - MOC) / (ORACLE - MOC)
tp  = 100 * len(pha) / max(1, sum(1 for q in qids if GOLD[q] & set(cur[q])))
print(f"bắt được {pct:.0f}% dư địa oracle ({(ORACLE-MOC)*100:+.2f} điểm)")
print("ĐÓNG BẤT KỂ ĐIỂM RÒNG: tỉ lệ phá > 3% — đúng chế độ hỏng đã giết Gemini"
      if tp > 3.0 else
      "ĐI TIẾP -> chạy đề thi top-8 (10,4h), nộp"        if d >= 1.50 and net >= 4 else
      "VÙNG MÙ -> KHÔNG mang lên đề thi"                 if d >= 0.70 else
      "ĐÓNG vai trò chấm lại danh sách ngắn")